# Bilge Pump System — Safety Analysis Notebook

## Overview
This notebook executes the full Safety/FMEA/RAAML/UQ verification pipeline
on top of the existing 4-layer SysML v2 model.

| Layer | File | Contents |
|---|---|---|
| Library | `Library.sysml` | All `part def`, `port def`, `attribute def` |
| Architecture | `Architecture.sysml` | 8 part usages + 11 connect statements |
| Requirements | `Requirements.sysml` | 4 `requirement def` blocks (BPS-REQ-001–004) |
| Analysis | `Analysis.sysml` | Physics `constraint def` + nominal `analysis def` |
| **RAAML** | `RAAML.sysml` | OMG RAAML v1.0 `metadata def` stereotypes |
| **Safety** | `Safety.sysml` | STPA Losses, Hazards, UCA `requirement def` blocks |
| **FMEA** | `FMEA.sysml` | FMEA constraint defs + 4 negative-test `analysis def` |
| **UQ** | `UQ.sysml` | N=10 parametric uncertainty sweep `analysis def` blocks |

## API Server
Uses the **SST public SysML v2 API** — same endpoint as Verification.ipynb.

| Item | Value |
|---|---|
| Endpoint | `http://sysml2.intercax.com:9000` |
| Managed by | SysML Submission Team (public service) |
| Auth | None |

## Execution Order (import chain)
```
RAAML → Library → Architecture → Requirements → Analysis → Safety → FMEA → UQ
```
Skipping any step will leave names unresolved downstream.
Re-run from the first skipped cell if you restart the kernel.

## Prerequisites
```bash
bash setup.sh   # installs Python deps
```
Cells 7–9 (STPA positive, FMEA negative tests, UQ sweep) implement
constraint evaluation **locally in Python** — the API stores the model;
Python evaluates the `require constraint` expressions against the bound values.

In [ ]:
# ==============================================================================
# Cell 1: RAAML metadata def compatibility check + connect to API + create project
#
# Checks that the Pilot API server supports 'metadata def' syntax (requires
# JAR version >= 2022-06). If the server returns a version that is too old,
# RAAML.sysml must be committed without metadata annotations (fallback mode).
#
# Also creates the project that all subsequent commits will target.
# ==============================================================================
import requests
import json
import os
import re

API_BASE  = "http://sysml2.intercax.com:9000"
SCRIPT_DIR = os.path.dirname(os.path.abspath("Safety.ipynb"))

# --- Health check ---
print(f"Connecting to SysML v2 API server at {API_BASE}...")
try:
    r = requests.get(f"{API_BASE}/projects", timeout=8)
    r.raise_for_status()
    print(f"  Server ready. Existing projects on server: {len(r.json())}")
except Exception as e:
    raise RuntimeError(
        f"Cannot reach {API_BASE}/projects — check internet/firewall.\nError: {e}"
    )

# --- Create a new project ---
r = requests.post(f"{API_BASE}/projects", json={
    "name": "BilgePump_SafetyAnalysis",
    "description": "Safety / FMEA / RAAML / UQ analysis — extends main 4-layer model"
})
r.raise_for_status()
project    = r.json()
PROJECT_ID = project["@id"]

print(f"\nProject created:")
print(f"  ID   : {PROJECT_ID}")
print(f"  Name : {project.get('name', 'BilgePump_SafetyAnalysis')}")
print(f"\n[RAAML NOTE] 'metadata def' requires SysML v2 Pilot JAR >= 2022-06.")
print(f"If RAAML.sysml commit fails with a parse error, the JAR may be older.")
print(f"Fallback: replace 'metadata def' with 'attribute def' in RAAML.sysml")
print(f"          and remove all '#Annotation {{ }}' blocks in Safety.sysml / FMEA.sysml.")

In [ ]:
# ==============================================================================
# Helper: commit_sysml_file
# Reads a .sysml file and POSTs it as a commit to the project.
# ==============================================================================
def commit_sysml_file(project_id: str, filepath: str, description: str) -> dict:
    """Read a .sysml file and POST it as a commit to the API server."""
    with open(filepath, "r") as f:
        content = f.read()
    payload = {
        "description": description,
        "changes": [
            {
                "@type": "TextualRepresentation",
                "body": content
            }
        ]
    }
    r = requests.post(
        f"{API_BASE}/projects/{project_id}/commits",
        json=payload
    )
    r.raise_for_status()
    result = r.json()
    print(f"  Committed: {os.path.basename(filepath)}  (commit id: {result.get('@id', 'n/a')})")
    return result

In [ ]:
# ==============================================================================
# Cell 3: Commit the full 8-layer model in import-chain order
#
# Order: RAAML → Library → Architecture → Requirements → Analysis
#        → Safety → FMEA → UQ
#
# RAAML must be first because Safety.sysml and FMEA.sysml import it.
# ==============================================================================
layers = [
    ("RAAML.sysml",        "RAAML metadata def library (OMG RAAML v1.0 stereotypes)"),
    ("Library.sysml",      "Library layer: part def, port def, attribute def"),
    ("Architecture.sysml", "Architecture layer: BilgePumpSystem with 8 part usages and 11 connections"),
    ("Requirements.sysml", "Requirements layer: 4 regulatory requirement defs (BPS-REQ-001 through 004)"),
    ("Analysis.sysml",     "Analysis layer: PumpFlowPhysics constraint + BilgePumpVerification"),
    ("Safety.sysml",       "Safety layer: STPA Losses, Hazards, 5 UCA requirement defs"),
    ("FMEA.sysml",         "FMEA layer: RPN/NPSH/ParallelFailureRate constraints + 4 negative tests"),
    ("UQ.sysml",           "UQ layer: N=10 parametric uncertainty sweep analysis defs"),
]

commits = {}
print("Committing 8-layer SysML v2 model...\n")
for i, (filename, description) in enumerate(layers, 1):
    filepath = os.path.join(SCRIPT_DIR, filename)
    print(f"[{i}/8] {filename}")
    commits[filename] = commit_sysml_file(PROJECT_ID, filepath, description)
    print()

print("All 8 layers committed successfully.")
print(f"Model is stored at: {API_BASE}/projects/{PROJECT_ID}")

In [ ]:
# ==============================================================================
# Cell 4: STPA POSITIVE TEST — nominal scenario
#
# Evaluates all 5 UCA safety requirement defs under nominal bindings.
# All 5 should be SATISFIED. This confirms the system architecture meets
# the safety constraints identified by STPA under normal operation.
#
# Python evaluates the require constraint expressions (same logic as verify.sh).
# ==============================================================================

# Nominal bindings (from Analysis.sysml BilgePumpVerification)
sys = {
    "sensor.waterLevel":          0.15,
    "controller.responseTime_s":  1.0,
    "controller.triggerLevel_m":  0.25,
    "pumpA.flowRate":             0.025,
    "pumpA.efficiency":           0.82,
    "pumpB.flowRate":             0.025,
    "pumpB.isRedundant":          True,
    "alarm.activationDelay_s":    0.5,
    "discharge.pipeLossFactor":   0.05,
}
designInflow = 0.030

# UCA safety requirement evaluations
uca_checks = {
    "UCA_001_ControllerNoActivatePumpA": (
        sys["controller.responseTime_s"] <= 5.0,
        f"responseTime_s = {sys['controller.responseTime_s']} <= 5.0"
    ),
    "UCA_002_SensorFailSilent": (
        sys["sensor.waterLevel"] >= 0.0 and sys["sensor.waterLevel"] <= 1.0,
        f"waterLevel = {sys['sensor.waterLevel']} in [0.0, 1.0]"
    ),
    "UCA_003_ControllerNoAlarm": (
        sys["alarm.activationDelay_s"] <= 2.0,
        f"activationDelay_s = {sys['alarm.activationDelay_s']} <= 2.0"
    ),
    "UCA_004_ControllerNoFailover": (
        sys["pumpB.isRedundant"] == True,
        f"pumpB.isRedundant = {sys['pumpB.isRedundant']}"
    ),
    "UCA_005_ControllerDelayedActivation": (
        sys["controller.responseTime_s"] <= 5.0,
        f"responseTime_s = {sys['controller.responseTime_s']} <= 5.0"
    ),
}

print("="*70)
print("STPA POSITIVE TEST — Nominal Scenario")
print("Bindings: nominal values from BilgePumpVerification")
print("="*70)
all_pass = True
for req_name, (result, detail) in uca_checks.items():
    status = "SATISFIED" if result else "VIOLATED"
    icon   = "✓" if result else "✗"
    if not result:
        all_pass = False
    print(f"  {icon} {req_name:45s}  {status}")
    print(f"      Detail: {detail}")

print()
if all_pass:
    print("STPA POSITIVE TEST: ALL 5 UCA REQUIREMENTS SATISFIED ✓")
else:
    print("STPA POSITIVE TEST: ONE OR MORE UCA REQUIREMENTS VIOLATED ✗")
    print("  Expected: all SATISFIED under nominal bindings.")

In [ ]:
# ==============================================================================
# Cell 5: FMEA NEGATIVE TESTS — 4 failure mode scenarios
#
# Injects failure mode attribute values derived from FMEA-BPS-003.
# Each scenario targets a specific failure mode and confirms that the
# expected requirements are VIOLATED.
#
# Scenarios:
#   FMEA-SC-001: Sensor stuck-at-zero (FM-S-001)   → expect UCA-002 + Discharge + Alarm VIOLATED
#   FMEA-SC-002: Pump A cavitation  (FM-PA-002)     → expect Discharge VIOLATED
#   FMEA-SC-003: Power bus loss     (FM-PB-001)     → expect Redundancy VIOLATED
#   FMEA-SC-004: Controller hang    (FM-C-001)      → expect UCA-001 + Discharge + Alarm VIOLATED
# ==============================================================================

def evaluate_reqs(bindings: dict, design_inflow: float = 0.030) -> dict:
    """Evaluate all requirement constraints for a given bindings dict."""
    s = bindings
    q_net = ((s.get("pumpA.flowRate", 0) + s.get("pumpB.flowRate", 0))
             * s.get("pumpA.efficiency", 0)
             * (1.0 - s.get("discharge.pipeLossFactor", 0)))
    return {
        "WaterLevelRequirement":          s.get("sensor.waterLevel", 0)    <= 0.3,
        "PumpRedundancyRequirement":       s.get("pumpB.isRedundant", False) == True,
        "AlarmResponseRequirement":        s.get("alarm.activationDelay_s", 9999) <= 2.0,
        "DischargeCapacityRequirement":    q_net >= design_inflow,
        "UCA_001_ControllerNoActivatePumpA": s.get("controller.responseTime_s", 9999) <= 5.0,
        "UCA_002_SensorFailSilent":        0.0 <= s.get("sensor.waterLevel", -1) <= 1.0,
        "UCA_003_ControllerNoAlarm":       s.get("alarm.activationDelay_s", 9999) <= 2.0,
        "UCA_004_ControllerNoFailover":    s.get("pumpB.isRedundant", False) == True,
        "_q_net": q_net,
    }

def print_fmea_scenario(name, fm_id, rpn, bindings, expected_violations):
    results = evaluate_reqs(bindings)
    q_net   = results.pop("_q_net")
    print(f"\n{'─'*70}")
    print(f"  {name}")
    print(f"  Failure Mode: {fm_id}   RPN = {rpn}  (threshold = 100)")
    print(f"  Q_net = {q_net:.4f} m³/s  (design inflow = 0.030 m³/s)")
    print(f"{'─'*70}")
    scenario_pass = True
    for req, sat in results.items():
        status = "SATISFIED" if sat else "VIOLATED"
        expected_violated = req in expected_violations
        correct = (sat and not expected_violated) or (not sat and expected_violated)
        icon = "✓" if correct else "✗ UNEXPECTED"
        if not correct:
            scenario_pass = False
        print(f"    {icon:14s} {req}: {status}")
    print(f"  → Scenario outcome: {'CORRECT (expected violations confirmed)' if scenario_pass else 'UNEXPECTED RESULT'}")
    return scenario_pass

print("="*70)
print("FMEA NEGATIVE TESTS")
print("="*70)

all_scenarios_correct = True

# FMEA-SC-001: Sensor stuck-at-zero
all_scenarios_correct &= print_fmea_scenario(
    "FMEA-SC-001: Sensor Stuck-At-Zero (FM-S-001)", "FM-S-001", 240,
    bindings={
        "sensor.waterLevel": 0.0, "controller.responseTime_s": 1.0,
        "pumpA.flowRate": 0.0, "pumpA.efficiency": 0.82,
        "pumpB.flowRate": 0.0, "pumpB.isRedundant": True,
        "discharge.pipeLossFactor": 0.05, "alarm.activationDelay_s": 999.0,
    },
    expected_violations=["DischargeCapacityRequirement", "AlarmResponseRequirement",
                         "UCA_003_ControllerNoAlarm"]
)

# FMEA-SC-002: Pump A cavitation
all_scenarios_correct &= print_fmea_scenario(
    "FMEA-SC-002: Pump A Cavitation (FM-PA-002)", "FM-PA-002", 210,
    bindings={
        "sensor.waterLevel": 0.15, "controller.responseTime_s": 1.0,
        "pumpA.flowRate": 0.025, "pumpA.efficiency": 0.40,  # cavitation
        "pumpB.flowRate": 0.025, "pumpB.isRedundant": True,
        "discharge.pipeLossFactor": 0.05, "alarm.activationDelay_s": 0.5,
    },
    expected_violations=["DischargeCapacityRequirement"]
)

# FMEA-SC-003: Power bus loss
all_scenarios_correct &= print_fmea_scenario(
    "FMEA-SC-003: Power Bus Loss / Redundancy Lost (FM-PB-001)", "FM-PB-001", 72,
    bindings={
        "sensor.waterLevel": 0.15, "controller.responseTime_s": 1.0,
        "pumpA.flowRate": 0.025, "pumpA.efficiency": 0.82,
        "pumpB.flowRate": 0.025, "pumpB.isRedundant": False,  # bus lost
        "discharge.pipeLossFactor": 0.05, "alarm.activationDelay_s": 0.5,
    },
    expected_violations=["PumpRedundancyRequirement", "UCA_004_ControllerNoFailover"]
)

# FMEA-SC-004: Controller hang
all_scenarios_correct &= print_fmea_scenario(
    "FMEA-SC-004: Controller Software Hang (FM-C-001)", "FM-C-001", 140,
    bindings={
        "sensor.waterLevel": 0.28, "controller.responseTime_s": 9999.0,  # hung
        "pumpA.flowRate": 0.0, "pumpA.efficiency": 0.82,
        "pumpB.flowRate": 0.0, "pumpB.isRedundant": True,
        "discharge.pipeLossFactor": 0.05, "alarm.activationDelay_s": 9999.0,  # frozen
    },
    expected_violations=["DischargeCapacityRequirement", "AlarmResponseRequirement",
                         "UCA_001_ControllerNoActivatePumpA", "UCA_003_ControllerNoAlarm"]
)

print(f"\n{'='*70}")
print(f"FMEA NEGATIVE TESTS: {'ALL SCENARIOS CONFIRMED ✓' if all_scenarios_correct else 'ONE OR MORE UNEXPECTED RESULTS ✗'}")

In [ ]:
# ==============================================================================
# Cell 6: STPA NEGATIVE TESTS — 3 loss scenarios from STPA-BPS-003
#
# Each scenario injects UCA-causing fault conditions and confirms the
# expected UCA requirement defs are VIOLATED.
# ==============================================================================
import json

# Load scenario data from the ingested STPA scenarios file
stpa_scenarios_path = os.path.join(SCRIPT_DIR, "docs/ingested/hazards/stpa-scenarios.json")
with open(stpa_scenarios_path) as f:
    stpa_data = json.load(f)

print("="*70)
print("STPA NEGATIVE TESTS — Loss Scenarios")
print("="*70)

all_stpa_correct = True

for scenario in stpa_data["scenarios"]:
    # Build bindings dict from scenario
    bindings = {}
    for b in scenario["bindings"]:
        key = b["path"].replace("sys.", "")
        bindings[key] = b["value"]

    results = evaluate_reqs(bindings)
    q_net   = results.pop("_q_net")

    expected_v = scenario["expected_violations"]

    print(f"\n{'─'*70}")
    print(f"  {scenario['id']}: {scenario['name']}")
    print(f"  {scenario['description']}")
    print(f"  Q_net = {q_net:.4f} m³/s")
    print(f"{'─'*70}")

    scenario_correct = True
    for req, sat in results.items():
        status = "SATISFIED" if sat else "VIOLATED"
        expected_violated = any(v in req or req in v for v in expected_v)
        correct = (sat and not expected_violated) or (not sat and expected_violated)
        if not correct:
            scenario_correct = False
        icon = "✓" if correct else "✗ UNEXPECTED"
        print(f"    {icon:14s} {req}: {status}")

    print(f"  → {'CORRECT' if scenario_correct else 'UNEXPECTED RESULT'}")
    all_stpa_correct &= scenario_correct

print(f"\n{'='*70}")
print(f"STPA NEGATIVE TESTS: {'ALL SCENARIOS CONFIRMED ✓' if all_stpa_correct else 'UNEXPECTED RESULTS ✗'}")

In [ ]:
# ==============================================================================
# Cell 7: RELIABILITY METRICS — FMEA RPN and parallel failure rate
#
# Evaluates the FMEA constraint equations for all 8 failure modes.
# Highlights failure modes that exceed the RPN action threshold (100).
# Also computes system MTBF using the parallel redundancy equation.
# ==============================================================================

# Load FMEA data from ingested JSON
fmea_path = os.path.join(SCRIPT_DIR, "docs/ingested/fmea/pump-fmea-table.json")
with open(fmea_path) as f:
    fmea_data = json.load(f)

RPN_THRESHOLD = fmea_data["_meta"]["rpn_action_threshold"]

print("="*70)
print(f"FMEA RISK PRIORITY NUMBERS (action threshold = {RPN_THRESHOLD})")
print("="*70)
print(f"  {'ID':<12} {'Component':<20} {'Failure Mode':<35} {'S':>2} {'O':>2} {'D':>2} {'RPN':>4} {'Action?'}")
print(f"  {'─'*12} {'─'*20} {'─'*35} {'─':>2} {'─':>2} {'─':>2} {'─':>4} {'─'*8}")

for fm in fmea_data["failure_modes"]:
    rpn    = fm["severity"] * fm["occurrence"] * fm["detection"]
    action = "ACTION" if rpn >= RPN_THRESHOLD else "-"
    flag   = " ◄" if rpn >= RPN_THRESHOLD else ""
    fm_short = (fm["failure_mode"][:33] + ".") if len(fm["failure_mode"]) > 34 else fm["failure_mode"]
    print(f"  {fm['id']:<12} {fm['component']:<20} {fm_short:<35} {fm['severity']:>2} {fm['occurrence']:>2} {fm['detection']:>2} {rpn:>4} {action}{flag}")

# Parallel failure rate / MTBF
lambda_A   = 1.5e-5   # failures/hour (from FMEA-BPS-002)
lambda_sys = lambda_A * lambda_A
mtbf_sys   = 1.0 / lambda_sys if lambda_sys > 0 else float('inf')

print(f"\n{'─'*70}")
print(f"PARALLEL REDUNDANCY RELIABILITY (ParallelRedundancyFailureRate constraint)")
print(f"  λ_A = λ_B = {lambda_A:.2e} failures/hour  (IEC 61508-6 Table B.5)")
print(f"  λ_sys = λ_A × λ_B = {lambda_sys:.2e} failures/hour")
print(f"  MTBF_sys = {mtbf_sys/8760:.1f} years  ({mtbf_sys:.0f} hours)")
print(f"  Both pumps must fail simultaneously for system loss.")

In [ ]:
# ==============================================================================
# Cell 8: UQ PARAMETRIC SWEEP — N=10 deterministic scenarios
#
# Loads the UQ configuration from docs/ingested/uq/pump-uq-config.json
# and evaluates DischargeCapacityRequirement for all 10 sweep points.
#
# Expected: 9 SATISFIED, 1 VIOLATED (UQ_Sweep_10 — combined 3σ extreme).
# A design margin plot (text-based) shows the Q_net distribution across sweeps.
# ==============================================================================

uq_path = os.path.join(SCRIPT_DIR, "docs/ingested/uq/pump-uq-config.json")
with open(uq_path) as f:
    uq_data = json.load(f)

DESIGN_INFLOW = 0.030

print("="*70)
print("UQ PARAMETRIC SWEEP — DischargeCapacityRequirement")
print(f"Design inflow threshold: {DESIGN_INFLOW} m³/s")
print("="*70)
print(f"  {'Sweep ID':<18} {'Varied Param':<18} {'σ step':>7}  {'Q_net (m³/s)':>13}  {'Margin %':>9}  {'Result'}")
print(f"  {'─'*18} {'─'*18} {'─':>7}  {'─'*13}  {'─':>9}  {'─'*10}")

satisfied_count = 0
violated_count  = 0
q_net_values    = []

fixed = {b["sysml_path"].replace("sys.",""): b["value"] for b in uq_data["fixed_parameters"]}

for sp in uq_data["sweep_points"]:
    b     = sp["bindings"]
    qa    = b.get("sys.pumpA.flowRate",  fixed.get("pumpA.flowRate",  0.025))
    qb    = fixed.get("pumpB.flowRate", 0.025)
    eta   = b.get("sys.pumpA.efficiency", fixed.get("pumpA.efficiency", 0.82))
    lam   = b.get("sys.discharge.pipeLossFactor", fixed.get("discharge.pipeLossFactor", 0.05))
    q_net = (qa + qb) * eta * (1.0 - lam)
    margin_pct = (q_net / DESIGN_INFLOW - 1.0) * 100
    satisfied = q_net >= DESIGN_INFLOW
    status    = "SATISFIED" if satisfied else "VIOLATED ◄"
    if satisfied: satisfied_count += 1
    else:         violated_count  += 1
    q_net_values.append((sp["id"], q_net, satisfied))
    print(f"  {sp['id']:<18} {sp['varied_param']:<18} {sp['sigma_step']:>+7}  {q_net:>13.4f}  {margin_pct:>+8.1f}%  {status}")

print(f"\n  Summary: {satisfied_count}/10 SATISFIED, {violated_count}/10 VIOLATED")
print(f"  Nominal Q_net = {(0.025+0.025)*0.82*(1-0.05):.4f} m³/s  ({((0.025+0.025)*0.82*(1-0.05)/DESIGN_INFLOW-1)*100:.1f}% above threshold)")
print()

if satisfied_count == 9 and violated_count == 1:
    print("UQ SWEEP: EXPECTED RESULT ✓")
    print("  9 of 10 sweep points satisfy DischargeCapacityRequirement.")
    print("  Only the combined 3σ extreme (UQ_Sweep_10) violates — by 6.7%.")
    print("  Design is robust at any individual parameter 3σ deviation.")
    print("  ACTION: Review whether simultaneous 3σ deviation on all three")
    print("  parameters is credible given independent failure modes.")
    print("  If credible: add 10% flow reserve margin (increase pump capacity).")
elif satisfied_count == 10:
    print("UQ SWEEP: ALL 10 SATISFIED — design margin is wider than expected.")
else:
    print(f"UQ SWEEP: {violated_count} violations — review parameter distributions.")

In [ ]:
# ==============================================================================
# Cell 9: TRACEABILITY SUMMARY
#
# Lists all new SysML model elements added by the Safety/FMEA/UQ pipeline
# and their source documents. This provides the traceability entries that
# TraceabilityAgent will cross-reference against lib/traceability.json.
# ==============================================================================

traceability_entries = [
    # RAAML metadata defs
    {"construct": "metadata def", "name": "Hazard",           "package": "BilgePump::RAAML", "source_doc": "RAAML.sysml", "section": "—"},
    {"construct": "metadata def", "name": "Loss",             "package": "BilgePump::RAAML", "source_doc": "RAAML.sysml", "section": "—"},
    {"construct": "metadata def", "name": "UCA",              "package": "BilgePump::RAAML", "source_doc": "RAAML.sysml", "section": "—"},
    {"construct": "metadata def", "name": "FailureMode",      "package": "BilgePump::RAAML", "source_doc": "RAAML.sysml", "section": "—"},
    {"construct": "metadata def", "name": "FaultTree",        "package": "BilgePump::RAAML", "source_doc": "RAAML.sysml", "section": "—"},
    {"construct": "metadata def", "name": "SafetyRequirement","package": "BilgePump::RAAML", "source_doc": "RAAML.sysml", "section": "—"},
    # Safety.sysml
    {"construct": "requirement def", "name": "UCA_001_ControllerNoActivatePumpA", "package": "BilgePump::Safety", "source_doc": "STPA-BPS-002", "section": "3.1"},
    {"construct": "requirement def", "name": "UCA_002_SensorFailSilent",          "package": "BilgePump::Safety", "source_doc": "STPA-BPS-002", "section": "3.2"},
    {"construct": "requirement def", "name": "UCA_003_ControllerNoAlarm",         "package": "BilgePump::Safety", "source_doc": "STPA-BPS-002", "section": "3.3"},
    {"construct": "requirement def", "name": "UCA_004_ControllerNoFailover",      "package": "BilgePump::Safety", "source_doc": "STPA-BPS-002", "section": "3.4"},
    {"construct": "requirement def", "name": "UCA_005_ControllerDelayedActivation","package": "BilgePump::Safety", "source_doc": "STPA-BPS-002", "section": "3.5"},
    # FMEA.sysml
    {"construct": "attribute def",   "name": "FailureModeAttr",         "package": "BilgePump::FMEA", "source_doc": "FMEA-BPS-001", "section": "—"},
    {"construct": "constraint def",  "name": "RiskPriorityNumber",      "package": "BilgePump::FMEA", "source_doc": "FMEA-BPS-002", "section": "2.1"},
    {"construct": "constraint def",  "name": "ParallelRedundancyFailureRate", "package": "BilgePump::FMEA", "source_doc": "FMEA-BPS-002", "section": "2.2"},
    {"construct": "constraint def",  "name": "NPSHMarginCheck",         "package": "BilgePump::FMEA", "source_doc": "FMEA-BPS-002", "section": "2.3"},
    {"construct": "analysis def",    "name": "FMEA_SensorStuckAtZero",  "package": "BilgePump::FMEA", "source_doc": "FMEA-BPS-003", "section": "3.1"},
    {"construct": "analysis def",    "name": "FMEA_PumpACavitation",    "package": "BilgePump::FMEA", "source_doc": "FMEA-BPS-003", "section": "3.2"},
    {"construct": "analysis def",    "name": "FMEA_PowerBusLoss",       "package": "BilgePump::FMEA", "source_doc": "FMEA-BPS-003", "section": "3.3"},
    {"construct": "analysis def",    "name": "FMEA_ControllerHang",     "package": "BilgePump::FMEA", "source_doc": "FMEA-BPS-003", "section": "3.4"},
    # UQ.sysml
    {"construct": "attribute def",   "name": "UncertaintyBounds",        "package": "BilgePump::UQ", "source_doc": "UQ-BPS-001", "section": "—"},
    {"construct": "analysis def",    "name": "UQ_Sweep_01",              "package": "BilgePump::UQ", "source_doc": "UQ-BPS-001", "section": "sweep_points[0]"},
    {"construct": "analysis def",    "name": "UQ_Sweep_10",              "package": "BilgePump::UQ", "source_doc": "UQ-BPS-001", "section": "sweep_points[9]"},
]

print("="*70)
print("TRACEABILITY SUMMARY — New model elements (Safety/FMEA/UQ pipeline)")
print("="*70)
print(f"  {'Construct':<18} {'Name':<42} {'Source Doc':<15} {'Section'}")
print(f"  {'─'*18} {'─'*42} {'─'*15} {'─'*10}")
for e in traceability_entries:
    print(f"  {e['construct']:<18} {e['name']:<42} {e['source_doc']:<15} {e['section']}")

print(f"\n  Total new elements: {len(traceability_entries)}")
print(f"  All elements have source_doc populated — TraceabilityAgent gate criteria met.")
print(f"\n  NOTE: 'metadata def' entries (BilgePump::RAAML) are not yet scanned")
print(f"  by the current TraceabilityAgent (only scans 6 construct types).")
print(f"  Extend TraceabilityAgent scan pattern to include 'metadata def' in a")
print(f"  follow-on task, or accept RAAML annotations as non-traceable for now.")

## Results Summary

| Phase | Test | Expected Result |
|---|---|---|
| STPA Positive | 5 UCA requirement defs under nominal bindings | All 5 SATISFIED |
| FMEA Negative | 4 failure-mode scenarios | Violations confirmed per FMEA-BPS-003 |
| STPA Negative | 3 loss scenarios | Violations confirmed per STPA-BPS-003 |
| Reliability | RPN + MTBF | 5 failure modes exceed RPN=100 threshold |
| UQ Sweep | 10 parametric sweep points | 9 SATISFIED, 1 VIOLATED (combined 3σ extreme) |

## Engineering Actions from UQ

**UQ_Sweep_10 VIOLATED** — the combined 3σ worst-case (flowRate −3σ AND efficiency −3σ AND pipe loss +3σ)
violates `DischargeCapacityRequirement` by 6.7%.

**Decision required:** Is the combined 3σ extreme simultaneous deviation on all three independent parameters a credible scenario?
- If **YES**: Increase design inflow margin by 10% (uprate pump capacity from 25 L/s to 27.5 L/s per pump).
- If **NO** (independent failure modes cannot occur simultaneously): Document the independence assumption and close.

## Next Steps

1. Extend `TraceabilityAgent` to scan `metadata def` constructs (RAAML annotations)
2. Run `VerificationAgent` against this extended project to confirm TraceabilityAgent gate opens
3. Address UQ_Sweep_10 engineering action (see above)
4. Run `AllocationMapper` on `docs/ingested/allocations/functional-allocation.json` to add `allocate` and `satisfy` relationships
5. Verify all 9 empty `docs/ingested/` subdirs now have content — re-run `SysML Orchestrator`